# 03 - Feature Engineering, Split & Feature Selection

- Tao credit_history_length tu earliest_cr_line, fico_mid tu cap fico_range_low/high
- Time-based train/validation/test split theo issue_d
- **Chon bien (shortlist theo IV) - tinh CHI TREN TAP TRAIN**
- WOE-transform cac bien duoc chon

> Thay doi so voi ban dau (rui ro R5 trong sprint_1_review.md): buoc chon bien truoc day nam o
> notebook 02 va tinh IV tren toan bo vintage, tuc la da dung ca nhan cua ky test. Hau qua that:
> `revol_util` co IV 0.025 tren toan bo tap nhung chi 0.0146 tren train - duoi nguong 0.02 - nen
> le ra khong duoc vao model, va chinh no gay loi he so sai dau o notebook 04. Nay buoc chon bien
> duoc dat sau split va chi dung nhan cua tap train.


In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

import pandas as pd
from src.paths import DATA_RAW, DATA_INTERIM, DATA_PROCESSED, MODELS, REPORTS_FIGURES


## Feature engineering

In [2]:
import numpy as np
from src.features.clean import engineer_features, winsorize
from src.data.filter_vintage import assert_no_leakage

accepted = pd.read_parquet(DATA_INTERIM / 'accepted_vintage_2015_2017.parquet')

RAW_COLS = [
    'loan_amnt', 'emp_length', 'home_ownership', 'annual_inc', 'purpose', 'dti',
    'delinq_2yrs', 'earliest_cr_line', 'fico_range_low', 'fico_range_high',
    'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc',
    'addr_state',
]
PRICING_COLS = ['int_rate', 'grade']   # chi dung tinh Expected Net Return o notebook 05
TARGET = 'loan_status'

df = accepted[['issue_d'] + RAW_COLS + PRICING_COLS + [TARGET]].copy()
df['bad_flag'] = (df[TARGET] == 'Charged Off').astype(int)
df = engineer_features(df)  # emp_length_years, credit_history_length, fico_mid

# Tap bien UNG VIEN (chua chon loc) - shortlist se duoc tinh sau split, o cell ben duoi.
CANDIDATE_FEATURES = [
    c for c in RAW_COLS
    if c not in ('emp_length', 'earliest_cr_line', 'fico_range_low', 'fico_range_high')
] + ['emp_length_years', 'credit_history_length', 'fico_mid']

# Lop chan thu hai ngoai whitelist: chan bien hau-giai-ngan va bien quyet dinh cua LC.
# Can thiet khi mo rong candidate set o Sprint 2 - luc do khong doc tay tung ten bien duoc nua.
assert_no_leakage(CANDIDATE_FEATURES)
print(f'{len(CANDIDATE_FEATURES)} bien ung vien (da qua kiem tra leakage):')
print(CANDIDATE_FEATURES)
print('\nShape:', df.shape)
df.head()


16 bien ung vien (da qua kiem tra leakage):
['loan_amnt', 'home_ownership', 'annual_inc', 'purpose', 'dti', 'delinq_2yrs', 'inq_last_6mths', 'open_acc', 'pub_rec', 'revol_bal', 'revol_util', 'total_acc', 'addr_state', 'emp_length_years', 'credit_history_length', 'fico_mid']

Shape: (643917, 21)


,issue_d,loan_amnt,home_ownership,annual_inc,purpose,dti,delinq_2yrs,inq_last_6mths,open_acc,pub_rec,...,revol_util,total_acc,addr_state,int_rate,grade,loan_status,bad_flag,emp_length_years,credit_history_length,fico_mid
0,2015-12-01,3600.0,MORTGAGE,55000.0,debt_consolidation,5.91,0.0,1.0,7.0,0.0,...,29.7,13.0,PA,13.99,C,Fully Paid,0,10.0,148,677.0
1,2015-12-01,24700.0,MORTGAGE,65000.0,small_business,16.06,1.0,4.0,22.0,0.0,...,19.2,38.0,SD,11.99,C,Fully Paid,0,10.0,192,717.0
2,2015-12-01,11950.0,RENT,34000.0,debt_consolidation,10.20,0.0,0.0,5.0,0.0,...,68.4,6.0,GA,13.44,C,Fully Paid,0,4.0,338,692.0
3,2015-12-01,20000.0,MORTGAGE,180000.0,debt_consolidation,14.67,0.0,0.0,12.0,0.0,...,84.5,27.0,MN,9.17,B,Fully Paid,0,10.0,306,682.0
4,2015-12-01,20000.0,MORTGAGE,85000.0,major_purchase,17.61,1.0,0.0,8.0,0.0,...,5.7,15.0,SC,8.49,B,Fully Paid,0,10.0,202,707.0


## Time-based split

In [3]:
from scipy.stats import chi2_contingency

df_sorted = df.sort_values('issue_d')
n = len(df_sorted)
train_cutoff = df_sorted['issue_d'].iloc[int(n * 0.7)]
val_cutoff = df_sorted['issue_d'].iloc[int(n * 0.85)]

print(f'Train: issue_d < {train_cutoff.date()}')
print(f'Val:   {train_cutoff.date()} <= issue_d < {val_cutoff.date()}')
print(f'Test:  issue_d >= {val_cutoff.date()}')

train_df = df[df['issue_d'] < train_cutoff].copy()
val_df = df[(df['issue_d'] >= train_cutoff) & (df['issue_d'] < val_cutoff)].copy()
test_df = df[df['issue_d'] >= val_cutoff].copy()

for name, part in [('Train', train_df), ('Val', val_df), ('Test', test_df)]:
    print(f'{name}: {len(part):,} ({len(part) / n:.1%}), bad rate {part.bad_flag.mean():.2%}')

# Kiem tra bad rate on dinh giua cac giai doan (phat hien vintage effect) - PROPOSAL muc 5.4
contingency = pd.DataFrame({
    'bad': [train_df.bad_flag.sum(), val_df.bad_flag.sum(), test_df.bad_flag.sum()],
    'good': [
        len(train_df) - train_df.bad_flag.sum(),
        len(val_df) - val_df.bad_flag.sum(),
        len(test_df) - test_df.bad_flag.sum(),
    ],
}, index=['train', 'val', 'test'])
chi2, p_value, _, _ = chi2_contingency(contingency)
print(f'\nChi-square bad rate train/val/test: chi2={chi2:.2f}, p={p_value:.4f}')
print('-> Bad rate KHAC BIET co y nghia thong ke giua cac giai doan (vintage effect).' if p_value < 0.05
      else '-> Bad rate on dinh giua cac giai doan, khong co bang chung vintage effect ro ret.')

# Winsorize: fit bounds CHI TREN train, ap dung lai cho val/test - tranh leakage nguong cat
train_df, winsor_bounds = winsorize(train_df)
val_df, _ = winsorize(val_df, winsor_bounds)
test_df, _ = winsorize(test_df, winsor_bounds)
print('\nWinsorize bounds (fit tren train):', {k: (round(v[0], 2), round(v[1], 2))
                                                for k, v in winsor_bounds.items()})


Train: issue_d < 2016-08-01
Val:   2016-08-01 <= issue_d < 2017-03-01
Test:  issue_d >= 2017-03-01
Train: 439,698 (68.3%), bad rate 16.31%
Val: 99,015 (15.4%), bad rate 21.48%
Test: 105,204 (16.3%), bad rate 19.94%

Chi-square bad rate train/val/test: chi2=1916.23, p=0.0000
-> Bad rate KHAC BIET co y nghia thong ke giua cac giai doan (vintage effect).



Winsorize bounds (fit tren train): {'annual_inc': (17000.0, 265000.0), 'dti': (2.07, 38.82), 'revol_bal': (259.0, 103564.33)}


## Chon bien (shortlist theo IV) — tinh CHI TREN TAP TRAIN

Buoc nay truoc day nam o notebook 02 va tinh IV tren toan bo vintage. Dat dung o day — **sau split**
— dam bao nhan cua ky val/test khong tham gia vao bat ky quyet dinh nao cua model.

Nguong `IV > 0.02` theo Siddiqi (2006): `<0.02` khong huu ich, `0.02–0.1` yeu, `0.1–0.3` trung binh,
`0.3–0.5` manh, `>0.5` nghi ngo qua manh/leakage.


In [4]:
from optbinning import BinningProcess

numeric_candidates = [c for c in CANDIDATE_FEATURES if pd.api.types.is_numeric_dtype(train_df[c])]
categorical_candidates = [c for c in CANDIDATE_FEATURES if c not in numeric_candidates]

# Fit tren TRAIN de tinh IV phuc vu chon bien
selection_bp = BinningProcess(
    variable_names=CANDIDATE_FEATURES,
    categorical_variables=categorical_candidates,
)
selection_bp.fit(train_df[CANDIDATE_FEATURES], train_df['bad_flag'])

iv_train = (selection_bp.summary()[['name', 'dtype', 'iv', 'gini', 'n_bins']]
            .sort_values('iv', ascending=False).reset_index(drop=True))

IV_THRESHOLD = 0.02
iv_train['selected'] = iv_train['iv'] > IV_THRESHOLD
FEATURES = iv_train.loc[iv_train['selected'], 'name'].tolist()

suspect = iv_train.loc[iv_train['iv'] > 0.5, 'name'].tolist()
if suspect:
    print(f'CANH BAO - IV > 0.5, kiem tra lai co phai leakage khong: {suspect}')

print(f'{len(FEATURES)}/{len(CANDIDATE_FEATURES)} bien vuot nguong IV > {IV_THRESHOLD} tren TRAIN:')
print(FEATURES)

# Doi chieu voi IV tinh tren toan bo vintage (notebook 02) - cho thay bien nao doi ket qua
iv_full = pd.read_csv(REPORTS_FIGURES / 'iv_table.csv')[['name', 'iv']].rename(columns={'iv': 'iv_full_vintage'})
compare = iv_train.merge(iv_full, on='name', how='left')
compare['doi_ket_qua'] = np.where(
    (compare['iv'] > IV_THRESHOLD) != (compare['iv_full_vintage'] > IV_THRESHOLD), '<-- DOI', ''
)
print('\nDoi chieu IV train vs IV toan bo vintage:')
print(compare[['name', 'iv', 'iv_full_vintage', 'selected', 'doi_ket_qua']].to_string(index=False))

iv_train.to_csv(REPORTS_FIGURES / 'iv_table_train.csv', index=False)
DATA_INTERIM.mkdir(parents=True, exist_ok=True)
with open(DATA_INTERIM / 'shortlist_features.txt', 'w') as f:
    f.write('\n'.join(FEATURES))
print('\nDa luu reports/figures/iv_table_train.csv va data/interim/shortlist_features.txt')


8/16 bien vuot nguong IV > 0.02 tren TRAIN:
['fico_mid', 'dti', 'annual_inc', 'inq_last_6mths', 'home_ownership', 'emp_length_years', 'credit_history_length', 'revol_bal']

Doi chieu IV train vs IV toan bo vintage:
                 name        iv  iv_full_vintage  selected doi_ket_qua
             fico_mid  0.150085         0.147485      True            
                  dti  0.063926         0.061396      True            
           annual_inc  0.060243         0.053465      True            
       inq_last_6mths    0.0499         0.042609      True            
       home_ownership  0.047367         0.050719      True            
     emp_length_years  0.023482         0.026422      True            
credit_history_length  0.022695         0.019203      True     <-- DOI
            revol_bal  0.020078         0.010553      True     <-- DOI
              purpose  0.019636         0.017832     False            
           addr_state  0.018332         0.018976     False            
    

## WOE transform

In [5]:
import pickle

numeric_features = [c for c in FEATURES if pd.api.types.is_numeric_dtype(train_df[c])]
categorical_features = [c for c in FEATURES if c not in numeric_features]

# Fit CHI TREN train - khong duoc fit tren toan bo du lieu, neu khong bin edges se "nhin thay"
# thong tin tu val/test (tuong lai), pha vo tinh chat time-based split.
binning_process = BinningProcess(
    variable_names=FEATURES,
    categorical_variables=categorical_features,
)
binning_process.fit(train_df[FEATURES], train_df['bad_flag'])


def add_woe(df_part: pd.DataFrame) -> pd.DataFrame:
    woe = binning_process.transform(df_part[FEATURES], metric='woe')
    woe.columns = [c + '_woe' for c in woe.columns]
    return pd.concat([df_part.reset_index(drop=True), woe.reset_index(drop=True)], axis=1)


train_df = add_woe(train_df)
val_df = add_woe(val_df)
test_df = add_woe(test_df)

MODELS.mkdir(parents=True, exist_ok=True)
with open(MODELS / 'binning_process.pkl', 'wb') as f:
    pickle.dump(binning_process, f)

print('WOE features:', [c + '_woe' for c in FEATURES])
train_df.filter(like='_woe').describe().T


WOE features: ['fico_mid_woe', 'dti_woe', 'annual_inc_woe', 'inq_last_6mths_woe', 'home_ownership_woe', 'emp_length_years_woe', 'credit_history_length_woe', 'revol_bal_woe']


,count,mean,std,min,25%,50%,75%,max
fico_mid_woe,439698.0,0.056541,0.422977,-0.399951,-0.268068,-0.070817,0.370129,1.099121
dti_woe,439698.0,0.020982,0.248666,-0.527381,-0.147204,0.031726,0.265956,0.327645
annual_inc_woe,439698.0,0.020182,0.245142,-0.448780,-0.123438,0.000992,0.238043,0.461519
inq_last_6mths_woe,439698.0,0.015985,0.216316,-0.437336,-0.125133,0.173568,0.173568,0.173568
home_ownership_woe,439698.0,0.016108,0.218931,-0.215889,-0.215889,-0.057326,0.247112,0.247112
emp_length_years_woe,439698.0,0.040022,0.062012,-0.034072,-0.000065,0.006971,0.127054,0.127054
credit_history_length_woe,439698.0,0.007569,0.149682,-0.289441,-0.124460,0.051982,0.130606,0.197239
revol_bal_woe,439698.0,0.007046,0.145942,-0.168535,-0.073554,-0.023834,0.055099,0.357359


## Luu ra data/processed

In [6]:
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

# Giu lai TOAN BO bien ung vien (khong chi shortlist) de notebook 05 dung cho Customer Dashboard
# va business rules - vi du purpose/addr_state khong lot shortlist IV nhung van huu ich khi hien
# thi ho so khach hang. Cac bien nay KHONG duoc dua vao model (model chi dung cot *_woe).
KEEP_COLS = (
    ['issue_d', 'bad_flag', 'loan_status'] + PRICING_COLS
    + CANDIDATE_FEATURES + [c + '_woe' for c in FEATURES]
)
KEEP_COLS = list(dict.fromkeys(KEEP_COLS))
train_df[KEEP_COLS].to_parquet(DATA_PROCESSED / 'train.parquet', index=False)
val_df[KEEP_COLS].to_parquet(DATA_PROCESSED / 'val.parquet', index=False)
test_df[KEEP_COLS].to_parquet(DATA_PROCESSED / 'test.parquet', index=False)

print('Da luu train/val/test vao data/processed/')
print({'train': train_df[KEEP_COLS].shape, 'val': val_df[KEEP_COLS].shape,
       'test': test_df[KEEP_COLS].shape})
print('\nBien vao model (WOE):', [c + '_woe' for c in FEATURES])
print('Bien chi de tham chieu/dashboard:', [c for c in CANDIDATE_FEATURES if c not in FEATURES])


Da luu train/val/test vao data/processed/
{'train': (439698, 29), 'val': (99015, 29), 'test': (105204, 29)}

Bien vao model (WOE): ['fico_mid_woe', 'dti_woe', 'annual_inc_woe', 'inq_last_6mths_woe', 'home_ownership_woe', 'emp_length_years_woe', 'credit_history_length_woe', 'revol_bal_woe']
Bien chi de tham chieu/dashboard: ['loan_amnt', 'purpose', 'delinq_2yrs', 'open_acc', 'pub_rec', 'revol_util', 'total_acc', 'addr_state']
